In [1]:
# Imports
from collections import Counter
import json
import os
import pickle
import random
import warnings

import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
import spacy

from cwi.baseline import BaselineFeatureBasedCWIdentifier
from cwi.freqdist_classifier import FrequencyDistCWIdentifier
from cwi.sievebased_cwidentifier import SievebasedCWIdentifier
from cwi.tmu_cwidentifier import TMUCWIdentifier
from cwi.util.data_reader import read_sharedtask, read_mdr

warnings.filterwarnings("ignore")

## Demonstration der Klassifikatoren auf dem Datenset der CWI Shared Task 2018

Nachfolgend werden die verschiedenen Klassifikatoren vorgeführt.

Zunächst müssen die Training- und Testdaten eingelesen werden:

In [2]:
# Shared task data
train_instances, train_labels = read_sharedtask("data/sharedtask/German_Train.tsv")
test_instances, test_labels = read_sharedtask("data/sharedtask/German_Test.tsv")

Außerdem die Frequenzverteilungen auf den verschiedenen Korpora für die Feature-Generierung:

In [3]:
# Wikipedia frequency distribution
with open("data/fdists/wikipedia_fdist.pkl", "rb") as file_in:
    wikipedia_fdist = pickle.load(file_in)

# Wikinews frequency distribution
with open("data/fdists/wikinews_fdist.pkl", "rb") as file_in:
    wikinews_fdist = pickle.load(file_in)

# Lang-8 frequency distribution
with open("data/fdists/lang8_fdist.pkl", "rb") as file_in:
    lang8_fdist = pickle.load(file_in)

# Klexikon frequency distribution
with open("data/fdists/klexikon_fdist.pkl", "rb") as file_in:
    klexikon_fdist = pickle.load(file_in)

Zuletzt ein paar Hilfsmittel für die Vorverarbeitung:

In [4]:
spacy_model = spacy.load("de_core_news_md")

with open("data/lookups/de_lemma_lookup.json", encoding="utf-8") as file_in:
    lemma_dict = json.load(file_in)

Hiermit haben wir schließlich alle benötigten Bestandteile zusammen, um die Klassifikatoren zu instanziieren:

In [39]:
# Baseline
bl_cwi = BaselineFeatureBasedCWIdentifier(wikipedia_fdist, spacy_model)

# TMU
tmu_cwi = TMUCWIdentifier(wikipedia_fdist, wikinews_fdist, lang8_fdist, spacy_model)

# TMU (without Lang-8)
tmu_wo_lang8_cwi = TMUCWIdentifier(wikipedia_fdist, wikinews_fdist, {}, spacy_model)
tmu_wo_lang8_cwi.disable_feature("lang8_freq")
tmu_wo_lang8_cwi.disable_feature("lang8_prob")

# Sieve-based: Frequency-threshold sieve
sb_cwi_th = SievebasedCWIdentifier(spacy_model, lemma_dict)

# Sieve-based: Familiarity classification sieve
sb_cwi_cl = SievebasedCWIdentifier(spacy_model, lemma_dict)
sb_cwi_cl.disable_sieve("frequency")
sb_cwi_cl.enable_sieve("familiarity_classification")

Hiermit können wir nun anfangen, Trainings- und Testinstanzen zu generieren, die Klassifikatoren zu trainieren, Testvorhersagen zu machen und zu evaluieren.

Zunächst auf den Shared-Task-Daten:

In [40]:
# Training/preparing the classifiers
print("Baseline:")
bl_train_instances = bl_cwi.create_feature_vector_batch(train_instances)
bl_cwi.train(bl_train_instances, train_labels)

print("TMU:")
tmu_train_instances = tmu_cwi.create_feature_vector_batch(train_instances)
tmu_cwi.train(tmu_train_instances, train_labels)

print("TMU (ohne Lang-8):")
tmu_wo_lang8_train_instances = tmu_wo_lang8_cwi.create_feature_vector_batch(train_instances)
tmu_wo_lang8_cwi.train(tmu_wo_lang8_train_instances, train_labels)

# Setting up and training familiarity classification sieve
fd_s = FrequencyDistCWIdentifier([wikipedia_fdist, wikinews_fdist, lang8_fdist, klexikon_fdist])
fd_s_train_instances = fd_s.create_feature_vector_batch(train_instances)
fd_s.train(fd_s_train_instances, train_labels)
sb_cwi_cl.familiarity_classifier = fd_s
sb_cwi_cl.character_threshold = 9

# Setting character and frequency threshold for sieve-based classifier with frequency threshold sieve as well
sb_cwi_th.character_threshold = 9
sb_cwi_th.frequency_threshold = 1
# Setting up different corpus combinations for testing
fdists = {
    "Wikipedia": wikipedia_fdist,
    "Wikinews": wikinews_fdist,
    "Wikipedia + Wikinews + Lang-8": wikipedia_fdist + lang8_fdist + wikinews_fdist,
    "Wikipedia + Wikinews + Lang-8 + Klexikon": wikipedia_fdist + lang8_fdist + wikinews_fdist + klexikon_fdist,
    "Lang-8 + Klexi": lang8_fdist + klexikon_fdist
}

Baseline:


Feature Vector Generation: 100%|██████████| 6151/6151 [00:07<00:00, 807.56it/s] 


TMU:


Feature Vector Generation: 100%|██████████| 6151/6151 [00:07<00:00, 774.88it/s] 


TMU (ohne Lang-8):


Feature Vector Generation: 100%|██████████| 6151/6151 [00:07<00:00, 841.62it/s] 


In [41]:
# Predict test instances
bl_pred = bl_cwi.predict_raw_batch(test_instances)
tmu_pred = tmu_cwi.predict_raw_batch(test_instances)
tmu_wo_lang8_pred = tmu_wo_lang8_cwi.predict_raw_batch(test_instances)
sb_cl_pred, sb_cl_sieves = map(list, zip(*sb_cwi_cl.predict_raw_batch(test_instances, return_sieves=True)))

Feature Vector Generation: 100%|██████████| 959/959 [00:00<00:00, 972.39it/s] 


Anhand der Vorhersagen können nun die Evaluationsmetriken berechnet werden:

In [8]:
# Baseline results
print("---Baseline Results----------------")
print("Precision:\t", precision_score(test_labels, bl_pred, average="macro"))
print("Recall:\t\t", recall_score(test_labels, bl_pred, average="macro"))
print("F1:\t\t", f1_score(test_labels, bl_pred, average="macro"))

---Baseline Results----------------
Precision:	 0.7302200340025747
Recall:		 0.7137376373125068
F1:		 0.7187136672850958


In [9]:
# TMU results
print("---TMU Results---------------------")
print("Precision:\t", precision_score(test_labels, tmu_pred, average="macro"))
print("Recall:\t\t", recall_score(test_labels, tmu_pred, average="macro"))
print("F1:\t\t", f1_score(test_labels, tmu_pred, average="macro"))

---TMU Results---------------------
Precision:	 0.7368685267385331
Recall:		 0.7249347651545563
F1:		 0.7290960451977402


In [42]:
# TMU (without Lang-8) results
print("---TMU (without Lang-8 Results-----")
print("Precision:\t", precision_score(test_labels, tmu_wo_lang8_pred, average="macro"))
print("Recall:\t\t", recall_score(test_labels, tmu_wo_lang8_pred, average="macro"))
print("F1:\t\t", f1_score(test_labels, tmu_wo_lang8_pred, average="macro"))

---TMU (without Lang-8 Results-----
Precision:	 0.7349257884972171
Recall:		 0.7264379037261414
F1:		 0.729680853655283


In [10]:
# sb_cwi_th results
print("---Sieve-based (freq_th) Results---")
for combi_identifier, fdist in fdists.items():
    print()
    sb_cwi_th.frequency_dict = fdist
    print("Corpora used:", combi_identifier)
    sb_th_pred, sb_th_sieves = map(list, zip(*sb_cwi_th.predict_raw_batch(test_instances, return_sieves=True)))
    print("Precision:\t", precision_score(test_labels, sb_th_pred, average="macro"))
    print("Recall:\t\t", recall_score(test_labels, sb_th_pred, average="macro"))
    print("F1:\t\t", f1_score(test_labels, sb_th_pred, average="macro"))

---Sieve-based (freq_th) Results---

Corpora used: Wikipedia


Feature Vector Generation: 100%|██████████| 959/959 [00:01<00:00, 694.03it/s]


Precision:	 0.7505156241170821
Recall:		 0.7426941534980476
F1:		 0.7458161889384654

Corpora used: Wikinews


Feature Vector Generation: 100%|██████████| 959/959 [00:00<00:00, 200225.87it/s]


Precision:	 0.7489751132278097
Recall:		 0.7432529834677566
F1:		 0.7456644066233107

Corpora used: Wikipedia + Wikinews + Lang-8


...

Precision:	 0.7529221545579002
Recall:		 0.7444094193642568
F1:		 0.7477593660526721

Corpora used: Wikipedia + Wikinews + Lang-8 + Klexikon


Feature Vector Generation: 100%|██████████| 959/959 [00:00<00:00, 158603.27it/s]


Precision:	 0.7529221545579002
Recall:		 0.7444094193642568
F1:		 0.7477593660526721

Corpora used: Lang-8 + Klexi


Feature Vector Generation: 100%|██████████| 959/959 [00:00<00:00, 111249.52it/s]


Precision:	 0.750140977443609
Recall:		 0.7549701653224334
F1:		 0.7520830863883949


In [11]:
# sb_cwi_cl results
print("---Sieve-based (fam_cl) Results----")
print("Precision:\t", precision_score(test_labels, sb_cl_pred, average="macro"))
print("Recall:\t\t", recall_score(test_labels, sb_cl_pred, average="macro"))
print("F1:\t\t", f1_score(test_labels, sb_cl_pred, average="macro"))

---Sieve-based (fam_cl) Results----
Precision:	 0.7044465257336545
Recall:		 0.7091210174811138
F1:		 0.7061331170294263


Für den siebbasierten Klassifikator können wir zudem die Siebe
für alle als komplex gelabelten Instanzen anzeigen.

In [12]:
sb_th_pred = pd.Series(sb_th_pred)
sb_cl_pred = pd.Series(sb_cl_pred)

sb_th_sieves = pd.Series(sb_th_sieves)
sb_cl_sieves = pd.Series(sb_cl_sieves)

In [13]:
# Sieve-based classifier (frequency threshold)
# True positive
sb_th_tp_sieves = sb_th_sieves[test_labels + sb_th_pred == 2]
# False positive
sb_th_fp_sieves = sb_th_sieves[test_labels - sb_th_pred == -1]
# Sieve-based classifier (familiarity classifier)
# True positive
sb_cl_tp_sieves = sb_cl_sieves[test_labels + sb_cl_pred == 2]
# False positive
sb_cl_fp_sieves = sb_cl_sieves[test_labels - sb_cl_pred == -1]

Jetzt können wir uns anschauen, wie oft mit welcher Siebkombination Wörter korrekterweise als komplex bestimmt wurden (True Positive):

In [14]:
# Sieve-based classifier (frequency threshold)
sb_th_tp_sieves.value_counts()

[length, frequency]                                     115
[length]                                                100
[frequency]                                              51
[contains_punctuation_characters, frequency]              3
[is_nominalization]                                       2
[length, is_nominalization]                               1
[length, contains_punctuation_characters, frequency]      1
Name: count, dtype: int64

In [15]:
# Sieve-based classifier (familiarity classifier)
sb_cl_tp_sieves.value_counts()

[length]                                     136
[length, familiarity_classification]          79
[familiarity_classification]                  32
[contains_punctuation_characters]              3
[is_nominalization]                            2
[length, is_nominalization]                    1
[length, contains_punctuation_characters]      1
Name: count, dtype: int64

Ebenso können wir uns anschauen, wie oft mit welcher Siebkombination Wörter fälschlicherweise als komplex bestimmt wurden (False Positive):

In [16]:
# Sieve-based classifier (frequency threshold)
sb_th_fp_sieves.value_counts()

[length]                       70
[frequency]                    31
[length, frequency]            20
[is_nominalization]             4
[length, is_nominalization]     1
Name: count, dtype: int64

In [17]:
# Sieve-based classifier (familiarity classifier)
sb_cl_fp_sieves.value_counts()

[length]                                67
[familiarity_classification]            55
[length, familiarity_classification]    23
[is_nominalization]                      4
[length, is_nominalization]              1
Name: count, dtype: int64

In [18]:
# Overview Sieve-based classifier (frequency threshold)
print("Number of True Positives:\t", len(sb_th_tp_sieves))
print("Number of False Positives:\t", len(sb_th_fp_sieves))
print("Number of True Negatives:\t", sum(test_labels + sb_th_pred == 0))
print("Number of False Negatives:\t", sum(test_labels - sb_th_pred == 1))

Number of True Positives:	 273
Number of False Positives:	 126
Number of True Negatives:	 457
Number of False Negatives:	 103


In [19]:
# Overview Sieve-based classifier (familiarity classifier)
print("Number of True Positives:\t", len(sb_cl_tp_sieves))
print("Number of False Positives:\t", len(sb_cl_fp_sieves))
print("Number of True Negatives:\t", sum(test_labels + sb_cl_pred == 0))
print("Number of False Negatives:\t", sum(test_labels - sb_cl_pred == 1))

Number of True Positives:	 254
Number of False Positives:	 150
Number of True Negatives:	 433
Number of False Negatives:	 122


Nachfolgend sind noch einige Annotationen zufälliger Sätze, vorgenommen mit den verschiedenen Klassifikatoren, dargestellt. Dabei hilft uns die folgende Funktion:

In [20]:
def visualize_annotation(sentence, spans):
    span_starts = Counter(i[0] for i in spans)
    span_ends = Counter(i[1] for i in spans)
    final_str = ""
    for c, char in enumerate(sentence):
        if c in span_ends:
            final_str += "]" * span_ends[c]
        if c in span_starts:
            final_str += "[" * span_starts[c]
        final_str += char
    return final_str

In [43]:
# Get all complex instances according to test set annotation
test_instance_series = pd.Series(test_instances)
gt_complex = test_instance_series[test_labels == 1]

Die Folgende Zelle kann wiederholt ausgeführt werden, um unterschiedliche Sätze zu testen.

In [37]:
sent = random.choice(test_instances)[0]

# Display test label annotation
print("Annotation according to test set annotation:")
gt_spans = [(i[1], i[2]) for i in filter(lambda x: x[0] == sent, gt_complex)]
print(visualize_annotation(sent, gt_spans) + "\n")

# Display baseline prediction
bl_instances, bl_pred = bl_cwi.annotate_text(sent)
bl_spans = [(i[1], i[2]) for i, p in zip(bl_instances, bl_pred) if p]
print(visualize_annotation(sent, bl_spans) + "\n")

# Display TMU prediction
print("Annotation according to TMU prediction:")
tmu_instances, tmu_pred = tmu_cwi.annotate_text(sent)
tmu_spans = [(i[1], i[2]) for i, p in zip(tmu_instances, tmu_pred) if p]
print(visualize_annotation(sent, tmu_spans) + "\n")

# Display sieve-based prediction (frequency threshold)
print("Annotation according to sieve-based prediction (frequency threshold):")
sb_th_instances, sb_th_pred = sb_cwi_th.annotate_text(sent)
sb_th_spans = [(i[1], i[2]) for i, p in zip(sb_th_instances, sb_th_pred) if p]
print(visualize_annotation(sent, sb_th_spans) + "\n")

# Display sieve-based prediction (familiarity classification)
print("Annotation according to sieve-based prediction (familiarity classification):")
sb_cl_instances, sb_cl_pred = sb_cwi_cl.annotate_text(sent)
sb_cl_spans = [(i[1], i[2]) for i, p in zip(sb_cl_instances, sb_cl_pred) if p]
print(visualize_annotation(sent, sb_cl_spans))

Annotation according to test set annotation:
Nun muss Taiwans Präsident [Chen Shui-bian] nach der Wahl am [Samstag] weitere vier Jahre ohne eine Mehrheit im Parlament regieren .



Feature Vector Generation: 100%|██████████| 21/21 [00:00<00:00, 1898.12it/s]


Nun muss Taiwans Präsident Chen [Shui-bian] nach der Wahl am Samstag weitere vier Jahre ohne eine Mehrheit im Parlament regieren .

Annotation according to TMU prediction:


Feature Vector Generation: 100%|██████████| 21/21 [00:00<00:00, 4551.72it/s]


Nun muss Taiwans Präsident Chen Shui-bian nach der Wahl am Samstag weitere vier Jahre ohne eine Mehrheit im Parlament regieren .

Annotation according to sieve-based prediction (frequency threshold):


Feature Vector Generation: 100%|██████████| 21/21 [00:00<00:00, 10180.35it/s]


Nun muss Taiwans Präsident Chen [Shui-bian] nach der Wahl am Samstag weitere [vier] Jahre ohne eine Mehrheit im Parlament regieren .

Annotation according to sieve-based prediction (familiarity classification):


Feature Vector Generation: 100%|██████████| 21/21 [00:00<00:00, 8544.86it/s]


Nun muss Taiwans Präsident Chen Shui-bian nach der Wahl am Samstag weitere [vier] Jahre ohne eine Mehrheit im Parlament regieren .
